# Week 1 - Day 3 Lab: Data & Matrix Manipulation
In this lab, you'll work with a realistic weather dataset. You'll use **Pandas** to explore and clean the data, and **NumPy** to perform matrix operations.

**Dataset:** `hourly_weather_10_days.csv` (10 days of hourly weather data)

## Step 1: Load the Data
- Use Pandas to load the CSV file
- Display the first few rows
- Check the number of rows and columns

In [2]:
# TODO: Load the data into a DataFrame
import pandas as pd

# Replace the file path if needed
df = pd.read_csv('hourly_weather_10_days.csv')
df.head()

,timestamp,temperature_C,humidity_%,wind_speed_kmph,pressure_hPa,visibility_km
0,2023-03-01 00:00:00,16.6,74.4,5.7,1012.5,9.5
1,2023-03-01 01:00:00,16.2,78.5,5.0,1012.1,10.3
2,2023-03-01 02:00:00,15.3,73.3,4.7,NaN,11.1
3,2023-03-01 03:00:00,15.8,72.4,1.3,1005.0,8.9
4,2023-03-01 04:00:00,20.9,70.6,6.8,1016.3,9.8


## Step 2: Basic Exploration
- Check column names and data types
- Display basic statistics using `.describe()`
- Count missing values in each column

In [34]:
# TODO: Explore the DataFrame
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   timestamp        240 non-null    datetime64[ns]
 1   temperature_C    240 non-null    float64       
 2   humidity_%       240 non-null    float64       
 3   wind_speed_kmph  240 non-null    float64       
 4   pressure_hPa     240 non-null    float64       
 5   visibility_km    240 non-null    float64       
 6   day              240 non-null    object        
 7   hour             240 non-null    int32         
dtypes: datetime64[ns](1), float64(5), int32(1), object(1)
memory usage: 14.2+ KB
None


In [32]:
print(df.describe())

                 timestamp  temperature_C  humidity_%  wind_speed_kmph  \
count                  240     240.000000  240.000000       240.000000   
mean   2023-03-05 23:30:00      21.315789   66.795982        10.105310   
min    2023-03-01 00:00:00      11.500000   47.800000         1.300000   
25%    2023-03-03 11:45:00      18.975000   61.650000         6.800000   
50%    2023-03-05 23:30:00      21.500000   66.795982        10.105310   
75%    2023-03-08 11:15:00      23.825000   72.150000        13.400000   
max    2023-03-10 23:00:00      28.700000   88.100000        17.800000   
std                    NaN       3.334242    7.911399         3.823509   

       pressure_hPa  visibility_km        hour  
count    240.000000     240.000000  240.000000  
mean    1011.884753       9.989474   11.500000  
min      998.100000       6.800000    0.000000  
25%     1009.225000       9.300000    5.750000  
50%     1011.884753       9.989474   11.500000  
75%     1014.800000      10.700000   17

In [33]:
print(df.isna().sum())

timestamp          0
temperature_C      0
humidity_%         0
wind_speed_kmph    0
pressure_hPa       0
visibility_km      0
day                0
hour               0
dtype: int64


## Step 3: Handle Missing Values
- Drop or fill missing values
- Justify your approach (e.g., fill with mean, forward fill, etc.)

In [4]:
# TODO: Fill missing values
# Justification: Each numeric column is missing its values, so
# dropping rows isn't necessary. Filling with the column mean fill is a simple method
# so standard approach that keeps the overall distribution (average, spread)
# roughly intact without discarding any data. It's not perfect for hourly
# time-series data (it flattens hour-to-hour variation), but it's a
# reasonable, easy-to-justify baseline for this dataset.

numeric_cols = ['temperature_C', 'humidity_%', 'wind_speed_kmph', 'pressure_hPa', 'visibility_km']

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

# confirm no missing values remain
print(df.isna().sum())

timestamp          0
temperature_C      0
humidity_%         0
wind_speed_kmph    0
pressure_hPa       0
visibility_km      0
dtype: int64


## Step 4: Data Analysis
- Calculate daily average temperature
- Find max, min, mean for each metric
- Which hour of the day is the most humid on average?

In [28]:
# TODO: Perform analysis

# Make sure timestamp is a real datetime, then extract day and hour
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['day'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour

# 1. Daily average temperature
daily_avg_temp = df.groupby('day')['temperature_C'].mean()
print("Daily average temperature:")
print(daily_avg_temp)

Daily average temperature:
day
2023-03-01    21.263158
2023-03-02    21.258991
2023-03-03    21.304825
2023-03-04    21.425658
2023-03-05    21.529825
2023-03-06    21.858333
2023-03-07    21.179825
2023-03-08    20.947807
2023-03-09    20.792325
2023-03-10    21.597149
Name: temperature_C, dtype: float64


In [26]:
# 2. Max, min, mean for each metric
metrics = ['temperature_C', 'humidity_%', 'wind_speed_kmph', 'pressure_hPa', 'visibility_km']
summary_stats = df[metrics].agg(['max', 'min', 'mean'])
print("\nSummary stats per metric:")
print(summary_stats)


Summary stats per metric:
      temperature_C  humidity_%  wind_speed_kmph  pressure_hPa  visibility_km
max       28.700000   88.100000         17.80000   1027.000000      12.600000
min       11.500000   47.800000          1.30000    998.100000       6.800000
mean      21.315789   66.795982         10.10531   1011.884753       9.989474


In [27]:
# 3. Which hour of the day is most humid on average
avg_humidity_by_hour = df.groupby('hour')['humidity_%'].mean()
most_humid_hour = avg_humidity_by_hour.idxmax()
print(f"\nMost humid hour on average: {most_humid_hour}:00, avg humidity = {avg_humidity_by_hour.max():.2f}%")


Most humid hour on average: 1:00, avg humidity = 78.42%


## Step 5: NumPy Matrix Exercises
Convert relevant DataFrame columns into NumPy arrays and perform matrix operations.

In [6]:
# TODO: Extract temperature and wind_speed as NumPy arrays
import numpy as np

temp = df['temperature_C'].to_numpy()
wind = df['wind_speed_kmph'].to_numpy()

### a) Reshape into matrix form
- Assume each row is a day
- Reshape temperature into a (10, 24) matrix
- Calculate daily min, max, and mean using axis-based operations

In [29]:
# TODO: Reshape and aggregate
temp_matrix = temp.reshape((10, 24))
# Write functions to find min, max, mean across rows
# axis=1 means we are iterating over one value per row (per day)
daily_min = temp_matrix.min(axis=1)
daily_max = temp_matrix.max(axis=1)
daily_mean = temp_matrix.mean(axis=1)

print("Daily min:", daily_min)
print("Daily max:", daily_max)
print("Daily mean:", daily_mean)

Daily min: [14.7 15.7 13.6 15.9 12.4 15.5 15.3 13.5 14.3 11.5]
Daily max: [28.2 28.7 25.7 27.1 24.9 26.2 25.9 26.  27.1 28.5]
Daily mean: [21.26315789 21.25899123 21.30482456 21.42565789 21.52982456 21.85833333
 21.17982456 20.94780702 20.79232456 21.59714912]


### b) Normalize the temperature matrix
- Subtract the mean and divide by std deviation
- Do it manually using NumPy functions

In [11]:
# TODO: Normalize temp_matrix
def normalize(temp_matrix):
    return (temp_matrix - temp_matrix.mean()) / temp_matrix.std()

# Apply it to temp_matrix
normalized_temp = normalize(temp_matrix)
print(normalized_temp)

[[-1.4173072  -1.53752522 -1.80801577 -1.65774325 -0.12496347 -0.15501798
   0.44607213  0.35590862 -0.03479995  2.06901543  0.          1.28759828
   1.22748927  0.62639917  1.88868839  0.74661719  0.20563609  0.71656268
   0.53623565  0.2356906  -0.93643512 -0.57578105 -0.48561753 -1.98834281]
 [-1.62768874 -1.68779775 -1.11676215 -0.39545402  0.20563609 -0.03479995
   0.41601763  0.05536356  0.68650818  2.21928795 -0.24518149  1.13732576
   0.2356906   0.95699873  0.38596312  0.17558158  0.77667169  0.
   0.86683521  0.20563609 -0.42550852 -0.75610808 -1.44736171 -0.99654413]
 [-2.31894237 -1.68779775 -0.8462716  -1.62768874  1.31765279 -0.12496347
  -0.15501798 -0.18507248  0.29579961  1.31765279  1.04716224 -0.03479995
   1.25754378  0.17558158  0.32585411  1.01710774  0.92694422  0.
   0.59634466  0.92694422  0.41601763 -0.12496347 -0.81621709 -1.77796127]
 [-1.3872527  -0.78616259 -1.02659863 -1.59763424  0.05536356  0.2356906
   1.73841587  0.71656268  0.17558158  0.86683521  1

### c) Apply custom mask/filter
- Create a mask for wind speed > 15 kmph
- Use it to extract high-wind readings

In [31]:
# TODO: Create boolean mask and filter wind speeds
mask = wind > 15
print("Mask :", mask)

Mask : [False False False False False False False False False False False  True
  True  True False False False False False False False False False False
 False False False False False False False False False False  True False
 False False False False False False False False False False False False
 False False False False False False False False False  True  True  True
  True False False  True  True False False False False False False False
 False False False False False False False False False False False  True
  True False False False False False False False False False False False
 False False False False False False False False False  True False  True
  True False  True False False False False False False False False False
 False False False False False False False False False False False False
 False  True False False False False False False False False False False
 False False False False False False False False False False  True  True
 False False False False False False False F

In [30]:
high_wind = wind[mask]
print("High wind readings:", high_wind)

High wind readings: [17.6 16.  16.5 16.3 16.7 15.8 17.8 15.1 16.3 15.2 17.  15.9 15.6 15.8
 15.4 15.6 16.3 15.3 16.2 16.9 15.3 15.2 15.5 17.4 17.4 15.4 15.4 16.5
 17.  15.7]


## Final Challenge: Write Your Own Function
Write a function `daily_summary(matrix)` that takes a NumPy matrix of shape (10, 24) and returns a summary dictionary for each day.

In [20]:
# TODO: Write and test your function
def daily_summary(matrix):
    list = []
    for i, row in enumerate(matrix):
        calculate_readings = {'day': i+1, 'min': row.min(), 'max': row.max(), 'mean': row.mean()}
        list.append(calculate_readings)
    return list

# Example usage:
summaries = daily_summary(normalized_temp)
for i in summaries:
    print(i)

{'day': 1, 'min': np.float64(-1.988342805934924), 'max': np.float64(2.0690154257461266), 'mean': np.float64(-0.015818160747294068)}
{'day': 2, 'min': np.float64(-1.6877977517363276), 'max': np.float64(2.219287952845425), 'mean': np.float64(-0.017070431806454883)}
{'day': 3, 'min': np.float64(-2.31894236555338), 'max': np.float64(1.3176527902496358), 'mean': np.float64(-0.003295450155686025)}
{'day': 4, 'min': np.float64(-1.627688740896608), 'max': np.float64(1.7384158661276714), 'mean': np.float64(0.03302041055997782)}
{'day': 5, 'min': np.float64(-2.6795964305916953), 'max': np.float64(1.0772167468907585), 'mean': np.float64(0.06432718703899824)}
{'day': 6, 'min': np.float64(-1.7479067625760467), 'max': np.float64(1.467925317348934), 'mean': np.float64(0.16305887370336183)}
{'day': 7, 'min': np.float64(-1.8080157734157658), 'max': np.float64(1.3777618010893549), 'mean': np.float64(-0.040863581930510505)}
{'day': 8, 'min': np.float64(-2.3489968709732394), 'max': np.float64(1.4078163065

## ✅ Submit your notebook once complete.
- Add comments where necessary

In [25]:
open('hourly_weather_10_days.csv', 'w').write("new row\n")

8